# Step 3 — Build the Historical Weather Panel

**Objective:** Create a county x day weather panel from NOAA ISD (2018-2024) and derive compound event flags (heatwave, compound heat-wind, compound triple).

**Output:** `data/processed/weather_panel_ercot.csv` and `weather_panel_caiso.csv`

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import RAW, PROCESSED, ERCOT_FIPS, CAISO_FIPS, HIST_START_YEAR, HIST_END_YEAR
from src.data.noaa_isd import build_weather_panel
from src.analysis.compound_flags import add_weather_flags


## 3.1 Load county geometries (from TIGER/Line, built in Step 1)

In [2]:
import geopandas as gpd

tiger_path = RAW['hifld'] / 'tl_2023_us_county' / 'tl_2023_us_county.shp'
if tiger_path.exists():
    counties = gpd.read_file(tiger_path)
    counties['fips'] = counties['GEOID'].astype(str).str.zfill(5)
    ercot_gdf = counties[counties['fips'].isin(set(ERCOT_FIPS))].copy()
    caiso_gdf = counties[counties['fips'].isin(set(CAISO_FIPS))].copy()
    print('Shapefiles loaded.')
else:
    print('Shapefile not found.')
    ercot_gdf = caiso_gdf = None


Shapefiles loaded.


## 3.2 Build ERCOT weather panel

ISD station files are downloaded from `s3://noaa-isd-pds/` and cached locally. Requires internet + `boto3`.

In [3]:
if ercot_gdf is not None:
    try:
        ercot_weather = build_weather_panel(
            region_fips=ERCOT_FIPS,
            county_gdf=ercot_gdf,
            cache_dir=RAW['noaa_isd'],
            year_range=(HIST_START_YEAR, HIST_END_YEAR),
        )
        print(f'ERCOT weather panel: {ercot_weather.shape}')
        print(ercot_weather.head())
    except Exception as e:
        print(f'Weather build failed: {e}')
        ercot_weather = None
else:
    ercot_weather = None


/burg-archive/home/mck2199/electric-grid-resilience/src/data/noaa_isd.py:107: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  joined = gpd.sjoin(gdf, county_gdf[["fips", "geometry"]], how="left", predicate="within")


ERCOT weather panel: (335042, 6)
                   tmax  tmin  heat_index_max  wind_speed_max     rh_min  \
fips  date                                                                 
48001 2018-01-01   0.50 -5.75             NaN            6.00  35.849180   
      2018-01-02  -1.50 -5.30             NaN            3.35  38.520339   
      2018-01-03   7.60 -7.40             NaN            3.70  27.026136   
      2018-01-04  10.05 -5.70             NaN            2.45  25.047417   
      2018-01-05  14.70 -0.35             NaN            3.20  27.016757   

                  precip_total  
fips  date                      
48001 2018-01-01           0.0  
      2018-01-02           0.0  
      2018-01-03           0.0  
      2018-01-04           0.0  
      2018-01-05           0.0  


## 3.3 Derive compound event flags

In [4]:
if ercot_weather is not None:
    ercot_weather = add_weather_flags(ercot_weather)
    flag_cols = ['heatwave_day', 'heatwave_event', 'compound_heat_wind',
                 'compound_heat_precip', 'compound_triple']
    present = [c for c in flag_cols if c in ercot_weather.columns]
    print('Flag rates (fraction of county-days):')
    print(ercot_weather[present].mean())


Flag rates (fraction of county-days):
heatwave_day            0.288540
heatwave_event          0.261713
compound_heat_wind      0.001895
compound_heat_precip    0.004402
compound_triple         0.000582
dtype: float64


## 3.3b Build CAISO weather panel


In [5]:
if caiso_gdf is not None:
    try:
        caiso_weather = build_weather_panel(
            region_fips=CAISO_FIPS,
            county_gdf=caiso_gdf,
            cache_dir=RAW['noaa_isd'],
            year_range=(HIST_START_YEAR, HIST_END_YEAR),
        )
        caiso_weather = add_weather_flags(caiso_weather)
        print(f'CAISO weather panel: {caiso_weather.shape}')
        print(caiso_weather.head())
    except Exception as e:
        print(f'CAISO weather build failed: {e}')
        caiso_weather = None
else:
    caiso_weather = None


/burg-archive/home/mck2199/electric-grid-resilience/src/data/noaa_isd.py:107: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  joined = gpd.sjoin(gdf, county_gdf[["fips", "geometry"]], how="left", predicate="within")


CAISO weather panel: (125885, 12)
                       tmax       tmin  heat_index_max  wind_speed_max  \
fips  date                                                               
06001 2018-01-01  14.366667   6.566667             NaN        3.100000   
      2018-01-02  14.866667   8.533333             NaN        3.314286   
      2018-01-03  13.900000   9.350000             NaN        3.685714   
      2018-01-04  16.800000  10.916667             NaN        3.971429   
      2018-01-05  16.350000  12.400000             NaN        5.342857   

                     rh_min  precip_total  heatwave_day  heatwave_event  \
fips  date                                                                
06001 2018-01-01  60.306292      0.000000             0               0   
      2018-01-02  58.358290      0.000000             0               0   
      2018-01-03  60.286022      3.166667             0               0   
      2018-01-04  67.978667     12.800000             0               0 

## 3.4 NOAA Storm Events supplement

Download bulk CSV from: https://www.ncei.noaa.gov/stormevents/ftp.jsp
Save to `data/raw/storm_events/`.

In [6]:
storm_files = list(RAW['storm_events'].glob('StormEvents_details*.csv'))
print(f'Found {len(storm_files)} Storm Events files')
if storm_files:
    se = pd.concat([pd.read_csv(f, low_memory=False) for f in storm_files], ignore_index=True)
    print(f'Total rows: {len(se):,}')
    print(se['EVENT_TYPE'].value_counts().head(10))


Found 7 Storm Events files


Total rows: 468,514
EVENT_TYPE
Thunderstorm Wind           126860
Hail                         58686
Flash Flood                  28211
High Wind                    27709
Winter Weather               26412
Drought                      25912
Flood                        21787
Winter Storm                 19973
Marine Thunderstorm Wind     17577
Heavy Snow                   16271
Name: count, dtype: int64


## 3.5 Save processed weather panels

In [7]:
if ercot_weather is not None:
    ercot_weather.to_csv(PROCESSED['weather_ercot'])
    print('Saved:', PROCESSED['weather_ercot'])
if caiso_weather is not None:
    caiso_weather.to_csv(PROCESSED['weather_caiso'])
    print('Saved:', PROCESSED['weather_caiso'])


Saved: /burg-archive/home/mck2199/electric-grid-resilience/data/processed/weather_panel_ercot.csv


Saved: /burg-archive/home/mck2199/electric-grid-resilience/data/processed/weather_panel_caiso.csv
